In [1]:
import subprocess


def run(command: str) -> None:
    """Run a WS4 command, printing what it printed."""
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    print((result.stdout or result.stderr).strip())
    result.check_returncode()


# The sample estate

`examples/estate` is a small self-contained estate: 64 blocks on an 8 x 8 grid, one clearcut
treatment that can be applied more than once, a seral feature that marks fresh openings, and a
road network with two mills. Everything below runs against it.

## The files

| File | What it holds |
| --- | --- |
| `recipe.xml` | two lanes, one clearcut operable from age 40, the volume curves, and a seral curve that is 1 while an opening is fresh |
| `fragments.csv` | 64 blocks, 8-12 ha, ages 0-100, all natural |
| `topology.csv` | the grid adjacency that makes adjacent openings a patch |
| `vertices.csv`, `segments.csv`, `destinations.csv`, `linkages.csv` | the road network and the two mills |
| `route.yaml`, `patches.yaml`, `eligibility.csv` | the route configuration and the patch account |


## Compile it

In [2]:
from pathlib import Path

from ws4.build.compiler import build_canonical_dataset

work = Path("tmp/examples/estate")
work.mkdir(parents=True, exist_ok=True)
dataset = build_canonical_dataset(
    "examples/estate/recipe.xml", "examples/estate/fragments.csv", work / "compiled"
)
print(dataset)

tmp/examples/estate/compiled


## Simulate a schedule: openings, and two mills

Cutting eight blocks in year 0 opens patches on the grid and ships the volume to whichever mill
is cheaper to reach.

In [3]:
import pandas as pd

schedule = pd.DataFrame({"block_id": [1, 2, 9, 10, 57, 58, 63, 64], "treatment": ["CC"] * 8, "period": [0] * 8})
(work / "schedule.csv").write_text(schedule.to_csv(index=False))

run("ws4 simulate --dataset tmp/examples/estate/compiled --schedule tmp/examples/estate/schedule.csv --years 10 --topology examples/estate/topology.csv --route-config examples/estate/route.yaml --out tmp/examples/estate/run")

Simulation complete → tmp/examples/estate/run


In [4]:
values = pd.read_csv(work / "run" / "account_values.csv")
print(values[values.year == 0].to_string(index=False))

 year                account        value
    0    feature.Seral.regen    81.900000
    0 feature.Volume.managed     0.000000
    0 feature.Volume.natural 71232.971429
    0   product.Harvested.CC  8793.814286
    0    product.Mill1.Yield  3423.514286
    0    product.Mill2.Yield  5370.300000
    0       route.Mill1.haul   684.702857
    0       route.Mill2.haul   749.115000
    0       route.Total.haul  1433.817857


`feature.Seral.regen` is the area in fresh openings — the criterion the patch account reads. The
two `product.Mill*` accounts and the two haul accounts say how the volume split, and
`route.Total.haul` is the duplicate account that adds them.

In [5]:
routed = pd.read_csv(work / "run" / "route_values.csv")
print(routed.groupby("destination")[["volume", "haul_cost"]].sum().to_string())
print()
print(routed[["block_id", "destination", "volume", "length_m", "haul_cost"]].head(4).to_string(index=False))

                        volume  haul_cost
destination                              
destination.MILL1  3423.514286        0.4
destination.MILL2  5370.300000        0.6

 block_id       destination      volume  length_m  haul_cost
        1 destination.MILL1 1918.714286    4000.0        0.2
        2 destination.MILL1 1504.800000    4000.0        0.2
        9 destination.MILL2 1272.000000    4000.0        0.2
       10 destination.MILL2 1932.000000    4000.0        0.2


Blocks near the west end go to `destination.MILL1`, blocks further east to `destination.MILL2`:
the choice is the least haul cost along the segments, not the distance to a mill.

## The openings themselves

In [6]:
metrics = pd.read_csv(work / "run" / "patch_metrics.csv")
print(metrics[metrics.year <= 2].to_string(index=False))

 year  patch_count  mean_patch_area  max_patch_area  le5  le10  le50  over
    0          3.0             27.3            42.9    0     0     3     0


## Search, and read the run

A volume floor plus a haul-cost ceiling behaves predictably, and shows the route accounts folding
into the objective.

In [7]:
(work / "targets.yaml").write_text(
    "targets:\n"
    "- account: product.Harvested.CC\n"
    "  year: 0\n"
    "  minimum: 20000\n"
    "  period_width: 10\n"
    "- account: route.Total.haul\n"
    "  year: 0\n"
    "  maximum: 3000\n"
    "  period_width: 10\n"
)

run("ws4 optimize --dataset tmp/examples/estate/compiled --targets tmp/examples/estate/targets.yaml --years 100 --iterations 20000 --route-config examples/estate/route.yaml --threads 1 --out tmp/examples/estate/search")

                              Optimization Summary                              
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ key                 ┃ value                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ incumbent_objective │ 0.0                                                    │
│ iterations          │ 20000                                                  │
│ seed                │ 42                                                     │
│ metaheuristic       │ sa                                                     │
│ starts              │ 1                                                      │
│ blocks_scheduled    │ 24                                                     │
│ operators           │ ['toggle', 'shift', 'swap', 'lns', 'cluster', 'merge', │
│                     │ 'split']                                               │
│ status              │ ok  

In [8]:
import json

summary = json.loads((work / "search" / "summary.json").read_text())
print({key: summary[key] for key in ("incumbent_objective", "blocks_scheduled", "iterations", "status")})
print()
print(pd.read_csv(work / "search" / "report_targets.csv").to_string(index=False))

{'incumbent_objective': 0.0, 'blocks_scheduled': 24, 'iterations': 20000, 'status': 'ok'}

             account  year    value  minimum  maximum  penalty status
product.Harvested.CC     0 1221.000  20000.0      NaN  18779.0  below
    route.Total.haul     0  213.675      NaN   3000.0      0.0    met


## What to look at next

* `docs/howto/route-products.rst` walks through the road tables and the route configuration.
* `docs/howto/patch-targets.rst` covers the patch accounts, the size classes and the shape-aware
  operators, and records that a patch **target** in a search currently misprices the classes: the
  search's view can read zero where a simulation shows openings, so a run's report is the side to
  trust until that is corrected.
* `docs/getting-started/architecture.rst` explains why the search prices a move instead of
  re-simulating.